# Notebook 02 — Multivariate Gaussians and Covariance

## Learning objectives
- Understand covariance matrices for vectors.
- Distinguish covariance and correlation.
- Generate correlated samples via Cholesky.
- Visualize covariance ellipses.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

np.random.seed(11)


## 1) Covariance matrix meaning

For $\mathbf{x}=[x\ y]^T$:

$$
\Sigma = \mathbb{E}[(\mathbf{x}-\mu)(\mathbf{x}-\mu)^T]
= \begin{bmatrix}\sigma_x^2 & \mathrm{cov}(x,y)\\ \mathrm{cov}(x,y) & \sigma_y^2\end{bmatrix}.
$$


In [ ]:
Sigma = np.array([[4.0, 1.5],[1.5, 2.0]])
print(Sigma)
print('eigvals:', np.linalg.eigvalsh(Sigma))


## 2) Correlation vs covariance

$$
\rho_{xy}=\frac{\mathrm{cov}(x,y)}{\sigma_x\sigma_y}
$$

Correlation is unitless; covariance has units.


In [ ]:
def corr_from_cov(S):
    return S[0,1]/np.sqrt(S[0,0]*S[1,1])

for rho in [0.0, 0.8, -0.8]:
    S = np.array([[9.0, rho*3*2],[rho*3*2, 4.0]])
    print(S)
    print('rho=', corr_from_cov(S))


## 3) Correlated samples with Cholesky

If $\Sigma=LL^T$ and $z\sim\mathcal{N}(0,I)$ then $x=\mu+Lz$ has covariance $\Sigma$.


In [ ]:
def sample_gaussian(mean, cov, N=2000):
    L = np.linalg.cholesky(cov)
    z = np.random.normal(size=(2, N))
    return (mean.reshape(2,1) + L @ z).T

mean = np.zeros(2)
cov = np.array([[3.0, 2.1],[2.1, 2.0]])
pts = sample_gaussian(mean, cov, N=4000)

print('empirical cov:')
print(np.cov(pts.T))
plt.figure(figsize=(6,6))
plt.scatter(pts[:,0], pts[:,1], s=5, alpha=0.25)
plt.axis('equal'); plt.show()


## 4) Interactive covariance ellipse (edit params and rerun)


In [ ]:
def ellipse_params(cov, n_std=2.0):
    w, v = np.linalg.eigh(cov)
    order = np.argsort(w)[::-1]
    w = w[order]; v = v[:,order]
    width, height = 2*n_std*np.sqrt(np.clip(w,0,None))
    angle = np.degrees(np.arctan2(v[1,0], v[0,0]))
    return width, height, angle

var_x, var_y, cov_xy = 4.0, 1.5, 1.3
cov = np.array([[var_x, cov_xy],[cov_xy, var_y]])
print('det=', np.linalg.det(cov))
if np.linalg.det(cov) > 0:
    pts = sample_gaussian(np.zeros(2), cov, N=2000)
    width, height, angle = ellipse_params(cov)
    fig, ax = plt.subplots(figsize=(6,6))
    ax.scatter(pts[:,0], pts[:,1], s=4, alpha=0.2)
    ax.add_patch(Ellipse((0,0), width, height, angle=angle, fill=False, lw=2, color='C3'))
    ax.set_aspect('equal'); plt.show()
else:
    print('Covariance is not positive definite.')


## Exercises

1. Build covariance for $\sigma_x=2$, $\sigma_y=5$, $ho=0.6$.
2. Interpret eigenvalues/eigenvectors geometrically.

## Common mistakes
- Non-PSD covariance matrices.
- Mixing covariance and correlation units.
- Forgetting transpose in `np.cov(samples.T)`.
